# Custom functions and outputs

You can record calls to a custom function on the database.

To do that just add the `record_call` decorator.

In [ ]:
from odyn import Database, Group, record_call

db = Database(r"tmp\test_server")

In [ ]:
# You need to have a Group/Database as the first input, and a * afterwards.
# All other inputs must be named and have default values.
# Each call to this function will be automatically recorded to the DB.

@record_call
def outcome_count(group, *, odor_ids=[], program_names=[]):
    programs = group.programs[group.programs.program_type.isin(program_names)]

    trials = group.trials[(group.trials.outcome != "na") & (group.trials.odor_id.isin(odor_ids))]
    trials = trials[["program_id", "odor_id", "outcome"]]

    result = trials.join(programs.program_type, on="program_id", how="inner")

    for odor_id in odor_ids:
        ax = result[result.odor_id == odor_id].pivot_table(
            index="program_type",
            columns="outcome",
            values="odor_id",
            aggfunc="count"
        ).plot(kind="barh")

        ax.set(title=f"Trials per Outcome (Odor {odor_id})", ylabel="Program", xlabel="Trial Count")

        # Reverse the labels so the order matches
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles[::-1], labels[::-1])

        fig_path = group.main_folder / f"test_plot.png"
        ax.get_figure().savefig(fig_path, bbox_inches="tight")

        # Add the output file to the DB
        group.add_output_file(fig_path)

In [ ]:
# You can run the function using the whole database
outcome_count(db, odor_ids=[17, 18], program_names=['fine 1', 'coarse 1', 'fine 2', 'coarse 2'])

In [ ]:
# Or you can run it using a specific group
group = db.groups[50]
outcome_count(group, odor_ids=[17], program_names=['fine 1', 'coarse 1', 'fine 2', 'coarse 2'])

In [ ]:
# This gives all calls to your function (from last to first)
# db.latest_calls("outcome_count") would get the corresponding table for a group
db.latest_calls("outcome_count")

In [ ]:
# This is the list of output files ever created
db.outputs

In [ ]:
# Same for the group
group.outputs